# 02 — Train DAN baseline

Prereqs: run `01_colab_setup.ipynb` first so deps are installed and `data/fer2013/` is populated.

**24h-mode (default):** trains DAN on **FER-2013** from ImageNet-init (no MS-Celeb-1M backbone needed) since RAF-DB requires a multi-day EULA. Target WAR ~67-72% (human upper bound on FER-2013 is ~65-68% per `report.md:1928`).

**Full-mode:** swap `--config` to `configs/dan_rafdb.yaml` once RAF-DB EULA + released checkpoint are in place; target WAR ≥ 87%.

This notebook:
1. Clones `yaoing/DAN` into `third_party/DAN/`.
2. Runs `python -m src.train --config configs/dan_fer2013.yaml`.
3. Evaluates `runs/dan_fer2013/best.pth` on the FER-2013 PrivateTest split.

In [ ]:
# === Bootstrap: mount Drive, clone repo, hydrate data, link runs/ to Drive ===
# Requires Colab "Secrets" entry GH_TOKEN with a fine-grained PAT for radudeaconu/fer.
import os
from pathlib import Path
from google.colab import drive, userdata

drive.mount('/content/drive')
if not Path('/content/fer').exists():
    try:
        os.environ['GH_TOKEN'] = userdata.get('GH_TOKEN')
    except Exception as e:
        from getpass import getpass
        print(f'[bootstrap] GH_TOKEN not in Colab Secrets ({type(e).__name__}).')
        print('[bootstrap] Create it via the key icon in the left sidebar:')
        print("[bootstrap]   name=GH_TOKEN, value=<fine-grained PAT for radudeaconu/fer>,")
        print("[bootstrap]   and toggle 'Notebook access' ON. Then re-run this cell.")
        print('[bootstrap] Or paste it below to use just for this session:')
        os.environ['GH_TOKEN'] = getpass('GH_TOKEN: ').strip()
    !git clone https://$GH_TOKEN@github.com/radudeaconu/fer.git /content/fer
%cd /content/fer
!pip install -q -r requirements.txt
%run scripts/colab_bootstrap.py

# DAN-specific: clone the original DAN repo for reference (third_party is gitignored).
import os
os.makedirs('third_party', exist_ok=True)
if not os.path.exists('third_party/DAN'):
    !git clone https://github.com/yaoing/DAN.git third_party/DAN


In [ ]:
# 24h-mode: no checkpoint to copy. DAN trains on FER-2013 from ImageNet-init.
# (For full-mode with RAF-DB, copy the released DAN checkpoint from Drive here
#  and set configs/dan_rafdb.yaml's pretrained_ckpt to its path.)
print('24h-mode: training DAN on FER-2013 from torchvision ImageNet ResNet-18 init.')

In [ ]:
!python -m src.train --config configs/dan_fer2013.yaml

In [ ]:
!python -m src.eval --config configs/dan_fer2013.yaml --ckpt runs/dan_fer2013/best.pth

In [ ]:
from IPython.display import Image
Image('runs/dan_fer2013/eval/confusion_matrix.png')